In [1]:
import gym 
import bauwerk
import numpy as np
from cfgs.parser import parse_cfg
from agent.sac import Agent as SAC
from cfgs.wrappers import dict_to_array
from cfgs.wrappers import ObsWrapper
from tqdm import tqdm


/Users/scottjeen/miniforge3/envs/odes/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
env = gym.make("bauwerk/SolarBatteryHouse-v0")
obs = env.reset()

In [4]:
eval_length = 24 * 30
seed = 1



env = gym.make("bauwerk/BuildDistB-v0")
obs = env.reset()
print(env.cfg.battery_size)
# mean random performance over 100 trials
random_trials = [evaluate_actions([env.action_space.sample() for _ in range(eval_length)], seed) for _ in range(100)]
random_std = np.std(random_trials)
p_rand = np.mean(random_trials)
# note: std here is between different trials (of multiple actions)
print(f"Avg reward with random actions: {p_rand:.4f} (standard deviation: {random_std:.4f})")

optimal_actions, _ = bauwerk.solve(env)
print(optimal_actions)
p_opt = evaluate_actions(optimal_actions[:eval_length], seed)
print(f"Avg reward (per step) with optimal actions: {p_opt:.4f}")

10.573081898100376
Avg reward with random actions: -0.8853 (standard deviation: 0.3859)
[ 0.     0.    -0.    ... -0.007 -0.    -0.   ]


AssertionError: 2.514984331902781e-10 (<class 'numpy.ndarray'>) invalid

In [15]:
from stable_baselines3.common.callbacks import BaseCallback

EVAL_LEN = 24 * 30

def eval_model(model, env):
    # Obtaining model actions and evaluating them
    model_actions = []
    obs = env.reset()
    for i in range(EVAL_LEN):
        action, _states = model.predict(obs)
        model_actions.append(action)
        obs, _, _, _ = env.step(action)

    p_model = evaluate_actions(model_actions[:EVAL_LEN], env)
    return p_model

def evaluate_actions(actions, seed):
    cum_reward = 0
    eval_env = gym.make("bauwerk/SolarBatteryHouse-v0")
    eval_obs = eval_env.reset()
    for action in actions:
        eval_obs, reward, done, info = eval_env.step(np.array(action, dtype=np.float32))
        
        cum_reward += reward
    
    print('Mean eval reward: {:.4f}'.format(cum_reward / len(actions)))
    return cum_reward / len(actions)

# callback for evaluating callback during training
class EvalCallback(BaseCallback):
    def __init__(self, eval_freq = 24*7, verbose=0):
        super().__init__(verbose)
        self.data = []
        self.eval_freq = eval_freq
        self.eval_env = gym.make("bauwerk/SolarBatteryHouse-v0")

    def _on_training_start(self) -> None:
        """
        This method is called before the first rollout starts.
        """
        self.data.append(eval_model(self.model, self.eval_env))

    def _on_step(self) -> bool:
        if self.num_timesteps % self.eval_freq == 0:
            self.data.append(eval_model(self.model, self.eval_env))

        return True

In [16]:
from stable_baselines3 import SAC

NUM_TRAIN_STEP = 24 * 365 * 2

model_sac = SAC(
    policy="MultiInputPolicy",
    env="bauwerk/SolarBatteryHouse-v0",
)
sac_callback = EvalCallback()
model_sac.learn(total_timesteps=NUM_TRAIN_STEP,callback=sac_callback)

p_model_sac = eval_model(model_sac, env)

print(f"Avg reward (per step) with model actions: {p_model_sac:.4f}")


Mean eval reward: -0.5095
Mean eval reward: -0.3068
Mean eval reward: -0.3449
Mean eval reward: -0.3096
Mean eval reward: -0.3068
Mean eval reward: -0.2483
Mean eval reward: -0.2860
Mean eval reward: -0.3062
Mean eval reward: -0.2967
Mean eval reward: -0.2938
Mean eval reward: -0.2614
Mean eval reward: -0.2658
Mean eval reward: -0.2498
Mean eval reward: -0.2419
Mean eval reward: -0.2328
Mean eval reward: -0.2313
Mean eval reward: -0.2131
Mean eval reward: -0.2129
Mean eval reward: -0.2320
Mean eval reward: -0.2040
Mean eval reward: -0.2394
Mean eval reward: -0.2164
Mean eval reward: -0.2042
Mean eval reward: -0.2036
Mean eval reward: -0.1844
Mean eval reward: -0.1799
Mean eval reward: -0.2141
Mean eval reward: -0.1870
Mean eval reward: -0.1901
Mean eval reward: -0.2068
Mean eval reward: -0.1819
Mean eval reward: -0.1749
Mean eval reward: -0.1704
Mean eval reward: -0.1794
Mean eval reward: -0.1784
Mean eval reward: -0.1822
Mean eval reward: -0.1738
Mean eval reward: -0.1765
Mean eval re

ValueError: too many values to unpack (expected 4)

In [23]:
model_sac.logger.train_critic_loss

AttributeError: 'Logger' object has no attribute 'train_critic_loss'

In [11]:
def evaluate(agent):
    eval_rewards = 0
    eval_env = gym.make("bauwerk/SolarBatteryHouse-v0")
    eval_env = ObsWrapper(eval_env)
    eval_obs = dict_to_array(eval_env.reset())
    print('eval size: {:.4f}'.format(eval_env.cfg.battery_size))
    for j in range(eval_steps):
        action, _ = agent.act(eval_obs, evaluate=True)
        eval_obs_, reward, done, _, _ = eval_env.step(action)
        eval_obs = eval_obs_
        eval_rewards += reward

    return eval_rewards / eval_steps # mean

In [ ]:
s

In [12]:
env = gym.make("bauwerk/SolarBatteryHouse-v0")
env = ObsWrapper(env)

cfg = parse_cfg()
agent = SAC(cfg=cfg, env=env, models_dir='tmp/')

episodes = 2
eval_interval = 24 * 7
eval_steps = 24 * 30
eval_episodes = 0

for i in tqdm(range(episodes)):
    ep_reward = 0
    evals = 0
    done = False
    obs = env.reset()
    obs = dict_to_array(obs)
    print('training size: {:.4f}'.format(env.cfg.battery_size))

    while not done:
        if agent.n_steps < cfg.seed_steps:
            action = env.action_space.sample()
        else:
            action, obs = agent.act(obs, evaluate=False)
        obs_, reward, done, _, _ = env.step(action)
        agent.memory.store_transition(obs, obs_, action, reward, done)
        obs = obs_
        agent.n_steps += 1
        ep_reward += reward
 
        if agent.n_steps > cfg.seed_steps:
            value_loss, actor_loss, critic_loss = agent.learn()

        if agent.n_steps % eval_interval == 0:
            eval_episodes += 1
            eval_reward = evaluate(agent)
            print('Eval episode {}, mean reward: {}'.format(eval_episodes, eval_reward))
            


  0%|                                                                                                                                       | 0/2 [00:00<?, ?it/s]

training size: 7.5000
eval size: 7.5000
Eval episode 1, mean reward: -0.15601225634721536
eval size: 7.5000
Eval episode 2, mean reward: -0.1568876485345552
eval size: 7.5000
Eval episode 3, mean reward: -0.16411726754154693
eval size: 7.5000
Eval episode 4, mean reward: -0.15855319001875615
eval size: 7.5000
Eval episode 5, mean reward: -0.16131693798069036
eval size: 7.5000
Eval episode 6, mean reward: -0.15355273276054504
eval size: 7.5000
Eval episode 7, mean reward: -0.15855319001875615
eval size: 7.5000
Eval episode 8, mean reward: -0.16356870381205227
eval size: 7.5000
Eval episode 9, mean reward: -0.16021024755349217
eval size: 7.5000
Eval episode 10, mean reward: -0.1687231527018513
eval size: 7.5000
Eval episode 11, mean reward: -0.1598809159903087
eval size: 7.5000
Eval episode 12, mean reward: -0.15896006899112966
eval size: 7.5000
Eval episode 13, mean reward: -0.15855676458118573
eval size: 7.5000
Eval episode 14, mean reward: -0.16031357806409183
eval size: 7.5000
Eval e

 50%|███████████████████████████████████████████████████████████████▌                                                               | 1/2 [01:32<01:32, 92.60s/it]

training size: 7.5000
eval size: 7.5000
Eval episode 53, mean reward: -0.15855319001875615
eval size: 7.5000
Eval episode 54, mean reward: -0.15855319001875615
eval size: 7.5000
Eval episode 55, mean reward: -0.15855319001875615
eval size: 7.5000
Eval episode 56, mean reward: -0.15855232534265976
eval size: 7.5000
Eval episode 57, mean reward: -0.15855319001875615
eval size: 7.5000
Eval episode 58, mean reward: -0.15704599539921119
eval size: 7.5000
Eval episode 59, mean reward: -0.15855319001875615
eval size: 7.5000
Eval episode 60, mean reward: -0.15855319001875615
eval size: 7.5000
Eval episode 61, mean reward: -0.15855319001875615
eval size: 7.5000
Eval episode 62, mean reward: -0.15855319001875615
eval size: 7.5000
Eval episode 63, mean reward: -0.15855319001875615
eval size: 7.5000
Eval episode 64, mean reward: -0.15855805909735257
eval size: 7.5000
Eval episode 65, mean reward: -0.16035066211867768
eval size: 7.5000
Eval episode 66, mean reward: -0.15855319001875615
eval size: 7

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [03:07<00:00, 93.82s/it]
